# Verify saved BRGM GRU forecasters

Independent reload and full-test re-evaluation of the saved BRGM GRU checkpoints.


In [ ]:
# Uncomment in Colab if needed
# !pip install numpy pandas scikit-learn torch tqdm


In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from IPython.display import display


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def safe_mkdir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


DEFAULT_DATASETS = (
    "piezo1", "piezo2", "piezo3", "piezo4", "piezo5", "piezo6", "piezo7", "piezo8",
    "piezo9", "piezo11", "piezo12", "piezo13", "piezo14", "piezo15", "piezo16",
    "piezo17", "piezo18",
)


@dataclass
class ExperimentConfig:
    datasets: Tuple[str, ...] = DEFAULT_DATASETS
    input_len: int = 336
    horizon: int = 24
    target_col: str = "level"
    train_ratio: float = 0.5
    val_ratio: float = 0.1
    batch_size: int = 128
    gru_hidden: int = 100
    gru_layers: int = 2
    change_feature_names: Tuple[str, ...] = ("PRELIQ_Q", "T_Q", "ETP_Q")
    seed: int = 42
    num_workers: int = 0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class SlidingWindowDataset(Dataset):
    """Same lazy sliding-window construction as the all-piezometers notebook."""

    def __init__(self, full_scaled: np.ndarray, start: int, end: int, input_len: int, horizon: int, target_idx: int) -> None:
        self.full_scaled = full_scaled
        self.start = start
        self.end = end
        self.input_len = input_len
        self.horizon = horizon
        self.target_idx = target_idx
        self.segment_len = max(end - start, 0)
        # Preserves the previous notebook's range(len(segment) - input_len - horizon).
        self.num_samples = max(self.segment_len - input_len - horizon, 0)

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        start_idx = self.start + idx
        x = self.full_scaled[start_idx : start_idx + self.input_len]
        y = self.full_scaled[start_idx + self.input_len : start_idx + self.input_len + self.horizon, self.target_idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), start_idx


@dataclass
class PreparedDataset:
    name: str
    csv_path: Path
    df: pd.DataFrame
    feature_cols: List[str]
    target_idx: int
    change_feature_idx: List[int]
    static_feature_idx: List[int]
    scaler: StandardScaler
    values_scaled: np.ndarray
    train_range: Tuple[int, int]
    val_range: Tuple[int, int]
    test_range: Tuple[int, int]
    train_ds: SlidingWindowDataset
    val_ds: SlidingWindowDataset
    test_ds: SlidingWindowDataset

    def inverse_target(self, y_scaled_1d: np.ndarray) -> np.ndarray:
        tmp = np.zeros((len(y_scaled_1d), len(self.feature_cols)), dtype=np.float32)
        tmp[:, self.target_idx] = y_scaled_1d
        inv = self.scaler.inverse_transform(tmp)
        return inv[:, self.target_idx]

    def make_loader(self, split: str, batch_size: int, shuffle: bool, seed: int, num_workers: int) -> DataLoader:
        dataset = {"train": self.train_ds, "val": self.val_ds, "test": self.test_ds}[split]
        generator = None
        if shuffle:
            generator = torch.Generator()
            generator.manual_seed(seed)
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator, num_workers=num_workers)


class GRUForecaster(nn.Module):
    """Same architecture as the saved best models in the all-piezometers notebook."""

    def __init__(self, input_size: int, hidden_size: int, num_layers: int, horizon: int):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0,
        )
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])


In [ ]:
def discover_dataset_files(search_root: Path, datasets: Sequence[str]) -> Tuple[List[Path], List[str]]:
    requested = {name.lower(): name for name in datasets}
    found: Dict[str, Path] = {}
    for csv_path in search_root.rglob("*.csv"):
        stem = csv_path.stem.lower()
        if stem in requested and stem not in found:
            found[stem] = csv_path
    ordered_paths = [found[name.lower()] for name in datasets if name.lower() in found]
    missing = [name for name in datasets if name.lower() not in found]
    return ordered_paths, missing


def load_piezo_dataframe(csv_path: Path, cfg: ExperimentConfig) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required_cols = {"date", cfg.target_col, *cfg.change_feature_names}
    missing_cols = sorted(required_cols - set(df.columns))
    if missing_cols:
        raise ValueError(f"{csv_path.name} is missing required columns: {', '.join(missing_cols)}")
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)


def prepare_dataset(csv_path: Path, cfg: ExperimentConfig) -> PreparedDataset:
    df = load_piezo_dataframe(csv_path, cfg)
    feature_cols = [col for col in df.columns if col != "date"]
    target_idx = feature_cols.index(cfg.target_col)
    change_feature_idx = [feature_cols.index(col) for col in cfg.change_feature_names]
    static_feature_idx = [idx for idx in range(len(feature_cols)) if idx not in change_feature_idx]

    values = df[feature_cols].values.astype(np.float32)
    n_rows = len(values)
    n_train = int(n_rows * cfg.train_ratio)
    n_val = int(n_rows * cfg.val_ratio)

    scaler = StandardScaler()
    scaler.fit(values[:n_train])
    values_scaled = scaler.transform(values).astype(np.float32)

    train_range = (0, n_train)
    val_range = (n_train - cfg.input_len - cfg.horizon, n_train + n_val)
    test_range = (n_train + n_val - cfg.input_len - cfg.horizon, n_rows)

    train_ds = SlidingWindowDataset(values_scaled, *train_range, cfg.input_len, cfg.horizon, target_idx)
    val_ds = SlidingWindowDataset(values_scaled, *val_range, cfg.input_len, cfg.horizon, target_idx)
    test_ds = SlidingWindowDataset(values_scaled, *test_range, cfg.input_len, cfg.horizon, target_idx)

    if len(train_ds) == 0 or len(val_ds) == 0 or len(test_ds) == 0:
        raise ValueError(f"{csv_path.name} is too short for input_len={cfg.input_len} and horizon={cfg.horizon}.")

    return PreparedDataset(
        name=csv_path.stem,
        csv_path=csv_path,
        df=df,
        feature_cols=feature_cols,
        target_idx=target_idx,
        change_feature_idx=change_feature_idx,
        static_feature_idx=static_feature_idx,
        scaler=scaler,
        values_scaled=values_scaled,
        train_range=train_range,
        val_range=val_range,
        test_range=test_range,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
    )


@torch.no_grad()
def predict_loader(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    preds, ys, starts, xs = [], [], [], []
    for xb, yb, start_idx in loader:
        pred = model(xb.to(device)).cpu().numpy()
        preds.append(pred)
        ys.append(yb.numpy())
        starts.append(start_idx.numpy())
        xs.append(xb.numpy())
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(preds), np.concatenate(starts)


def compute_regression_metrics(y_true_inv: np.ndarray, y_pred_inv: np.ndarray) -> Dict[str, float]:
    metrics = {
        "mae": float(mean_absolute_error(y_true_inv.ravel(), y_pred_inv.ravel())),
        "rmse": float(np.sqrt(mean_squared_error(y_true_inv.ravel(), y_pred_inv.ravel()))),
    }
    try:
        metrics["r2"] = float(r2_score(y_true_inv.ravel(), y_pred_inv.ravel()))
    except Exception:
        metrics["r2"] = float("nan")
    return metrics


def inverse_horizon(bundle: PreparedDataset, arr_2d: np.ndarray) -> np.ndarray:
    return np.vstack([bundle.inverse_target(row) for row in arr_2d])


## Configure paths



In [ ]:
from pathlib import Path

def find_project_root(start=None):
    """Locate the repository root from the current working directory."""
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "Data").is_dir() and (candidate / "Results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate CFE_for_MTS_Forecasting_Framework. "
        "Run this notebook from somewhere inside the cloned repository."
    )

PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)

SEARCH_ROOT = PROJECT_ROOT / "Data" / "piezos"
OUTPUT_DIR = PROJECT_ROOT / "Results" / "EDBT_piezo_GRU"
MODEL_ROOT = OUTPUT_DIR / "datasets"
RE_EVAL_OUTPUT_DIR = safe_mkdir(OUTPUT_DIR / "best_model_re_evaluation")

cfg = ExperimentConfig()
set_seed(cfg.seed)
device = torch.device(cfg.device)

print("Device:", device)
print("Search root:", SEARCH_ROOT.resolve())
print("Model root:", MODEL_ROOT.resolve())
print("Re-evaluation output:", RE_EVAL_OUTPUT_DIR.resolve())


In [ ]:
dataset_paths, missing_datasets = discover_dataset_files(SEARCH_ROOT.resolve(), cfg.datasets)

print(f"Found {len(dataset_paths)} dataset CSV files.")
if missing_datasets:
    print("Missing datasets:", missing_datasets)

pd.DataFrame({"dataset": [p.stem for p in dataset_paths], "csv_path": [str(p) for p in dataset_paths]})


In [ ]:
def find_saved_model_path(model_root: Path, dataset_name: str) -> Path:
    # Primary path used by the all-piezometers notebook.
    direct = model_root / dataset_name / "best_gru_forecaster.pth"
    if direct.exists():
        return direct

    # Fallback: search recursively in case your folder was moved or renamed.
    matches = sorted(model_root.rglob(f"{dataset_name}/best_gru_forecaster.pth"))
    if matches:
        return matches[0]

    matches = sorted(model_root.rglob("best_gru_forecaster.pth"))
    matches = [p for p in matches if p.parent.name.lower() == dataset_name.lower()]
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not find best_gru_forecaster.pth for {dataset_name} under {model_root}")


@torch.no_grad()
def load_and_evaluate_one(csv_path: Path, cfg: ExperimentConfig, model_root: Path) -> Dict[str, object]:
    bundle = prepare_dataset(csv_path, cfg)
    test_loader = bundle.make_loader(
        split="test",
        batch_size=cfg.batch_size,
        shuffle=False,
        seed=cfg.seed,
        num_workers=cfg.num_workers,
    )

    model = GRUForecaster(
        input_size=len(bundle.feature_cols),
        hidden_size=cfg.gru_hidden,
        num_layers=cfg.gru_layers,
        horizon=cfg.horizon,
    ).to(device)

    model_path = find_saved_model_path(model_root, bundle.name)
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    _, y_scaled, pred_scaled, starts = predict_loader(model, test_loader, device)
    y_inv = inverse_horizon(bundle, y_scaled)
    pred_inv = inverse_horizon(bundle, pred_scaled)
    metrics = compute_regression_metrics(y_inv, pred_inv)

    return {
        "dataset": bundle.name,
        "csv_path": str(csv_path),
        "model_path": str(model_path),
        "n_rows": len(bundle.df),
        "n_features": len(bundle.feature_cols),
        "features": ", ".join(bundle.feature_cols),
        "train_windows": len(bundle.train_ds),
        "val_windows": len(bundle.val_ds),
        "test_windows": len(bundle.test_ds),
        "test_mae": metrics["mae"],
        "test_rmse": metrics["rmse"],
        "test_r2": metrics["r2"],
        "first_test_start_idx": int(starts[0]) if len(starts) else None,
        "last_test_start_idx": int(starts[-1]) if len(starts) else None,
    }


In [ ]:
rows = []
errors = []

for csv_path in dataset_paths:
    try:
        print(f"Evaluating {csv_path.stem}...")
        rows.append(load_and_evaluate_one(csv_path, cfg, MODEL_ROOT))
    except Exception as exc:
        errors.append({"dataset": csv_path.stem, "error": repr(exc)})
        print(f"  ERROR: {exc}")

re_eval_df = pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)
errors_df = pd.DataFrame(errors)

re_eval_path = RE_EVAL_OUTPUT_DIR / "best_models_test_re_evaluation.csv"
re_eval_df.to_csv(re_eval_path, index=False)

print("Saved:", re_eval_path)
display(re_eval_df)

if not errors_df.empty:
    display(errors_df)


In [ ]:
# Optional: compare against the previous forecast_metrics.csv from the all-piezometers notebook.
previous_metrics_path = OUTPUT_DIR / "forecast_metrics.csv"

if previous_metrics_path.exists() and not re_eval_df.empty:
    previous = pd.read_csv(previous_metrics_path)
    print("Previous metrics columns:", list(previous.columns))
    display(previous.head())

    dataset_col = "dataset" if "dataset" in previous.columns else "name" if "name" in previous.columns else None
    metric_map = {
        "test_mae": ["test_mae", "mae", "test_mae_full"],
        "test_rmse": ["test_rmse", "rmse", "test_rmse_full"],
        "test_r2": ["test_r2", "r2", "test_r2_full"],
    }

    if dataset_col is not None:
        prev_small = previous.rename(columns={dataset_col: "dataset"}).copy()
        keep = ["dataset"]
        rename_cols = {}
        for target_name, candidates in metric_map.items():
            for c in candidates:
                if c in prev_small.columns:
                    keep.append(c)
                    rename_cols[c] = f"previous_{target_name}"
                    break
        prev_small = prev_small[keep].rename(columns=rename_cols)
        compare = re_eval_df.merge(prev_small, on="dataset", how="left")
        for m in ["test_mae", "test_rmse", "test_r2"]:
            pm = f"previous_{m}"
            if pm in compare.columns:
                compare[f"delta_{m}"] = compare[m] - compare[pm]
        compare_path = RE_EVAL_OUTPUT_DIR / "best_models_test_re_evaluation_comparison.csv"
        compare.to_csv(compare_path, index=False)
        print("Saved comparison:", compare_path)
        display(compare)
    else:
        print("Could not identify dataset column in previous metrics file.")
else:
    print("No previous forecast_metrics.csv found, or no re-evaluation rows produced.")


In [ ]:
# Quick summary table
if not re_eval_df.empty:
    summary = re_eval_df[["dataset", "test_windows", "test_mae", "test_rmse", "test_r2", "model_path"]].copy()
    display(summary.round({"test_mae": 4, "test_rmse": 4, "test_r2": 4}))


## What this notebook verifies

If all paths are correct and the architecture/preprocessing match the previous notebook, the test metrics should match the values previously reported for the saved best models, up to tiny numerical differences.

